# 🧪 Thai AI Assistant Playground (GPU Edition + Edge TTS)

Notebook นี้ออกแบบมาเพื่อทดสอบและโชว์ศักยภาพของแต่ละ Module (STT, LLM, TTS) โดยเน้นการใช้ **GPU** และ **Edge TTS** เพื่อเสียงที่ชัดเจน

**Hardware Requirements:**
- NVIDIA GPU (แนะนำ VRAM 4GB+ สำหรับ Whisper Small/Medium)
- RAM 8GB+
- Microphone (สำหรับทดสอบอัดเสียง)
- Internet Connection (สำหรับ Edge TTS)

**Components:**
1. **Speech-to-Text (STT):** `faster-whisper` (รันบน GPU)
2. **LLM (Brain):** `ollama` (Llama 3 / Typhoon)
3. **Text-to-Speech (TTS):** `edge-tts` (เสียงชัดแจ๋ว ภาษาไทย/อังกฤษ)

## 1. 🛠️ Installation & Setup
ติดตั้ง Library ที่จำเป็นและตรวจสอบ GPU

In [ ]:
# ติดตั้ง Library (ถ้ายังไม่ได้ติดตั้ง)
!pip install faster-whisper ollama transformers torch scipy sounddevice numpy matplotlib ipython edge-tts nest_asyncio

In [ ]:
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")

# ตรวจสอบ GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_allocated = torch.cuda.memory_allocated(0) / 1e9
    
    print("\n✅ GPU Detected:")
    print(f"   Name: {gpu_name}")
    print(f"   VRAM Total: {vram_total:.2f} GB")
    print(f"   VRAM Used: {vram_allocated:.2f} GB")
else:
    device = torch.device("cpu")
    print("\n⚠️ No GPU detected. Running on CPU (Might be slow).")

## 2. 🗣️ Speech-to-Text (STT) - `faster-whisper`
โหลดโมเดล Whisper ลง GPU และทดสอบแปลงเสียงเป็นข้อความ

In [ ]:
from faster_whisper import WhisperModel
import time
import os

# Configuration
model_size = "small" # ตัวเลือก: tiny, base, small, medium, large-v3
compute_type = "float16" if torch.cuda.is_available() else "int8"

print(f"⏳ Loading Whisper Model ('{model_size}') on {device} ({compute_type})...")
start_load = time.time()

# Load Model
stt_model = WhisperModel(model_size, device=str(device), compute_type=compute_type)

end_load = time.time()
print(f"✅ Model loaded in {end_load - start_load:.2f} seconds.")

In [ ]:
import sounddevice as sd
import scipy.io.wavfile as wav
import numpy as np

def record_audio(duration=5, filename="test_audio.wav", fs=44100):
    print(f"\n🎙️ Recording for {duration} seconds... (Speak Now!)")
    
    # Record audio
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='float32')
    sd.wait()  # Wait until recording is finished
    
    print("✅ Recording finished.")
    
    # Convert to 16-bit PCM for compatibility
    recording_int16 = (recording * 32767).astype(np.int16)
    
    # Save file
    wav.write(filename, fs, recording_int16)
    print(f"💾 Saved into file: {filename}")
    return filename

# record_audio(duration=5)

## 3. 🧠 LLM (Brain) - `ollama`
ทดสอบคุยกับ AI

In [ ]:
import ollama

def chat_with_ollama(prompt, model_name='llama3'):
    print(f"\n🤖 AI ({model_name}) is thinking...")
    
    response = ollama.chat(model=model_name, messages=[
        {'role': 'user', 'content': prompt},
    ])
    
    return response['message']['content']

# Test Chat
# print(chat_with_ollama("สวัสดีครับ"))

## 4. 🔊 High-Quality TTS - `edge-tts`
ใช้ Edge TTS เพื่อเสียงที่ชัดเจนกว่า (ต้องต่อเน็ต)

In [ ]:
import edge_tts
import nest_asyncio
import asyncio
from IPython.display import Audio, display

# Allow async in Notebook
nest_asyncio.apply()

async def text_to_speech_edge(text, lang='th'):
    # Voice Selection
    if lang == 'th':
        voice = "th-TH-PremwadeeNeural" # Female Thai
        # voice = "th-TH-NiwatNeural" # Male Thai
    else:
        voice = "en-US-AriaNeural" # Female English
        # voice = "en-US-GuyNeural" # Male English

    communicate = edge_tts.Communicate(text, voice)
    output_file = "output_edge.mp3"
    
    await communicate.save(output_file)
    print(f"🗣️ Generated Audio ({lang}): {text[:50]}...")
    
    return output_file

def play_audio(file_path):
    display(Audio(file_path, autoplay=True))

## 5. 🔄 Integrated Chat & Speak (Text -> LLM -> Audio)
พิมพ์คุยแล้วให้ AI พูดตอบกลับมาเลย! (Flow ที่คุณขอมา)

In [ ]:
# --- 🔴 ตั้งค่าคำถามตรงนี้ --- 
user_question = "แนะนำสถานที่ท่องเที่ยวในเชียงใหม่หน่อยครับ ขอแบบตื่นเต้นๆ"
language_mode = "th" # th = ไทย, en = อังกฤษ

# 1. ส่งคำถามไปหา LLM
print(f"👤 You: {user_question}")
ai_response = chat_with_ollama(user_question, model_name='llama3') # หรือเปลี่ยนเป็น 'scb10x/llama3.1-typhoon2-8b-instruct'

print(f"\n🤖 AI: {ai_response}")

# 2. แปลงคำตอบเป็นเสียง
audio_file = asyncio.run(text_to_speech_edge(ai_response, lang=language_mode))

# 3. เล่นเสียง
play_audio(audio_file)